# 🔥 Full RAG + Agentic RAG + LangGraph Multi-Agent Classroom Lab
### *Built with LlamaIndex · ChromaDB · Ollama · OpenRouter · Sentence-Transformers · LangGraph*

---

**Level:** Intermediate → Advanced  
**Duration:** 3–4 hours (full lab)  
**Goal:** Build a production-grade RAG system and a multi-agent LangGraph Resume Builder from scratch.

---

## 📚 Section 1 — Lab Introduction

### What is RAG?
**Retrieval-Augmented Generation (RAG)** combines:
- 🔍 **Retrieval** — Finding relevant documents from a knowledge base
- 🧠 **Augmentation** — Injecting retrieved context into the prompt
- ✍️ **Generation** — LLM produces grounded, factual answers

### 🏗️ Architecture
```
INDEXING: Docs → Chunk → Embed → ChromaDB
QUERY:    Query → Embed → Retrieve Top-K → LLM → Answer
AGENTIC:  Query → Agent (LLM) → Plan → Tools (RAG/Search) → Answer
```

## ⚙️ Section 2 — Environment Setup

In [ ]:
# ============================================================
# CELL 2.1 — Install all required packages
# ============================================================

!pip install -q llama-index llama-index-core
!pip install -q llama-index-llms-openai-like
!pip install -q llama-index-llms-ollama
!pip install -q llama-index-embeddings-ollama
!pip install -q llama-index-embeddings-huggingface
!pip install -q llama-index-vector-stores-chroma
!pip install -q llama-index-agent-openai
!pip install -q chromadb
!pip install -q pypdf
!pip install -q sentence-transformers
!pip install -q openai
!pip install -q requests
!pip install -q python-dotenv
!pip install -q langgraph
!pip install -q reportlab
!pip install -q pymupdf
!pip install -q python-docx

print('✅ All packages installed successfully!')

In [1]:
# ============================================================
# CELL 2.2 — Core imports
# ============================================================

import os
import sys
import json
import warnings
import textwrap
import subprocess
from pathlib import Path
from typing import TypedDict, List, Any, Optional

warnings.filterwarnings('ignore')

# LlamaIndex core
from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    Settings,
    Document,
)
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.response_synthesizers import get_response_synthesizer
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.agent import ReActAgent

# Vector Store
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore

# LangGraph
from langgraph.graph import StateGraph, END

print('✅ All imports successful!')
print(f'Python version: {sys.version}')

✅ All imports successful!
Python version: 3.13.13 (tags/v3.13.13:01104ce, Apr  7 2026, 19:25:48) [MSC v.1944 64 bit (AMD64)]


## 🤖 Section 3 — Load LLM (Ollama OR OpenRouter)

In [2]:
# ============================================================
# CELL 3.0 — Check if Ollama is running
# ============================================================
import requests as req

def check_ollama():
    try:
        r = req.get('http://localhost:11434/api/tags', timeout=3)
        if r.status_code == 200:
            models = [m['name'] for m in r.json().get('models', [])]
            print('✅ Ollama is running!')
            print(f'📦 Available models: {models if models else "No models pulled yet"}')
            return True, models
    except Exception:
        pass
    print('❌ Ollama not detected. Start with: "ollama serve" in terminal')
    print('📥 Then pull models: "ollama pull qwen2.5:7b" and "ollama pull nomic-embed-text"')
    return False, []

ollama_running, available_models = check_ollama()

✅ Ollama is running!
📦 Available models: ['nomic-embed-text:latest', 'phi4-mini:latest', 'qwen3:1.7b']


In [ ]:
%pip uninstall -y llama-index llama-index-core llama-index-llms-ollama
%pip install -U llama-index llama-index-core llama-index-llms-ollama


In [4]:
%pip show llama-index


Name: llama-index
Version: 0.14.22
Summary: Interface between LLMs and your data
Home-page: https://llamaindex.ai
Author: 
Author-email: Jerry Liu <jerry@llamaindex.ai>
License-Expression: MIT
Location: d:\Trainings\AI\abu dhabhi agent\.venv\Lib\site-packages
Requires: llama-index-core, llama-index-embeddings-openai, llama-index-llms-openai, nltk
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# ============================================================
# CELL 3.1 — OPTION A: Local LLM via Ollama
# ============================================================
from llama_index.llms.ollama import Ollama

OLLAMA_MODEL = "qwen3:1.7b"
OLLAMA_BASE_URL = "http://localhost:11434"
USE_OPENROUTER = False
llm_ollama = Ollama(
    model=OLLAMA_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0.1,
    request_timeout=120.0,
)

if ollama_running:
    try:
        test_resp = llm_ollama.complete(
            "Say 'Ollama ready now' in exactly 3 words."
        )
        print("Ollama LLM test:", test_resp)
        USE_OLLAMA = True
    except Exception as e:
        print("Ollama model error:", e)
        USE_OLLAMA = False
else:
    print("Skipping Ollama test — Ollama server not running.")
    USE_OLLAMA = False


In [ ]:
# ============================================================
# CELL 3.2 — OPTION B: OpenRouter (Cloud LLM)
# ============================================================
from llama_index.llms.openai_like import OpenAILike

# Set your key here OR in environment variable OPENROUTER_API_KEY
OPENROUTER_API_KEY = os.environ.get('OPENROUTER_API_KEY', 'YOUR_KEY_HERE')
OPENROUTER_MODEL =   'qwen/qwen-2.5-7b-instruct'

llm_openrouter = OpenAILike(
    model=OPENROUTER_MODEL,
    api_base='https://openrouter.ai/api/v1',
    api_key=OPENROUTER_API_KEY,
    temperature=0.1,
    max_tokens=2048,
    is_chat_model=True,
    default_headers={
        'HTTP-Referer': 'https://classroom.rag.lab',
        'X-Title': 'RAG Classroom Lab',
    }
)

if OPENROUTER_API_KEY and not OPENROUTER_API_KEY.endswith('YOUR_KEY_HERE'):
    try:
        test_resp = llm_openrouter.complete("Say 'OpenRouter ready!' in exactly 3 words.")
        print(f'✅ OpenRouter LLM test: {test_resp}')
        USE_OPENROUTER = True
    except Exception as e:
        print(f'⚠️  OpenRouter error: {e}')
        USE_OPENROUTER = False
else:
    print('⏭️  OpenRouter key not set. Set OPENROUTER_API_KEY environment variable.')
    USE_OPENROUTER = False

KeyboardInterrupt: 

In [ ]:
# ============================================================
# CELL 3.3 — Select Active LLM
# ============================================================

if USE_OPENROUTER:
    llm = llm_openrouter
    print(f'🌐 Using OpenRouter: {OPENROUTER_MODEL}')
elif USE_OLLAMA:
    llm = llm_ollama
    print(f'🏠 Using Ollama: {OLLAMA_MODEL}')
else:
    print('⚠️  No LLM available! Configure Ollama or OpenRouter.')
    class MockLLM:
        def complete(self, prompt):
            return type('R', (), {'text': '[MOCK] Configure Ollama or OpenRouter first'})()
    llm = MockLLM()

Settings.llm = llm
print('✅ LLM configured in LlamaIndex Settings')

🏠 Using Ollama: qwen3:1.7b
✅ LLM configured in LlamaIndex Settings


## 🧩 Section 5 — Embeddings

In [ ]:
# ============================================================
# CELL 5.1 — Setup Embedding Model
# ============================================================

embed_model = None
embed_model_name = None

# PRIMARY: Ollama nomic-embed-text
if ollama_running:
    try:
        from llama_index.embeddings.ollama import OllamaEmbedding
        embed_model = OllamaEmbedding(
            model_name='nomic-embed-text',
            base_url='http://localhost:11434',
            embed_batch_size=10,
        )
        test_vec = embed_model.get_text_embedding('test embedding')
        embed_model_name = 'nomic-embed-text (Ollama)'
        print(f'✅ Using Ollama: nomic-embed-text')
        print(f'   Embedding dimensions: {len(test_vec)}')
    except Exception as e:
        print(f'⚠️  Ollama embedding failed: {e}')
        embed_model = None

# FALLBACK: sentence-transformers
if embed_model is None:
    from llama_index.embeddings.huggingface import HuggingFaceEmbedding
    print('⏭️  Using HuggingFace sentence-transformers as fallback.')
    embed_model = HuggingFaceEmbedding(
        model_name='sentence-transformers/all-MiniLM-L6-v2',
        embed_batch_size=32,
    )
    test_vec = embed_model.get_text_embedding('test embedding')
    embed_model_name = 'all-MiniLM-L6-v2 (sentence-transformers)'
    print(f'✅ Using HuggingFace: all-MiniLM-L6-v2')
    print(f'   Embedding dimensions: {len(test_vec)}')

Settings.embed_model = embed_model
print(f'\n🎯 Active embedding model: {embed_model_name}')

✅ Using Ollama: nomic-embed-text
   Embedding dimensions: 768

🎯 Active embedding model: nomic-embed-text (Ollama)


## 🏗️ Section 12 — RAG-Powered Resume Builder
### Multi-Agent LangGraph Architecture

```
START
  │
  ▼
Load samplecv/ → Build ChromaDB Candidate KB
  │
  ▼
Resume Planner Agent (analyze template + KB)
  │
  ▼
Section Content Agent (per-section generation)
  │
  ▼
Validation Agent → PASS or FAIL → Retry Loop (max 2)
  │
  ▼
Resume Assembly Agent → final_resume.md
  │
  ▼
Quality Report Agent → resume_generation_report.md
  │
  ▼
END
```

In [27]:
# ============================================================
# CELL 12.2 — Build Candidate Knowledge Base in ChromaDB
# ============================================================

print('📖 Loading candidate CV documents...')


cv_reader = SimpleDirectoryReader(
    input_dir='./samplecv',
    required_exts=['.txt', '.pdf', '.docx'],
    recursive=True,
)
cv_documents = cv_reader.load_data()
print(f'✅ Loaded {len(cv_documents)} CV document(s)')

# Chunk documents
cv_splitter = SentenceSplitter(chunk_size=256, chunk_overlap=32)
cv_nodes = cv_splitter.get_nodes_from_documents(cv_documents)
print(f'✅ Created {len(cv_nodes)} chunks from CV documents')

CHROMA_DB_PATH = './chroma_db'
CHROMA_COLLECTION = 'rag_lab_candidate_kb'

chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

# Build ChromaDB collection for candidate KB
cv_collection = chroma_client.get_or_create_collection(
    name='candidate_kb',
    metadata={'hnsw:space': 'cosine'}
)
cv_vector_store = ChromaVectorStore(chroma_collection=cv_collection)
cv_storage_context = StorageContext.from_defaults(vector_store=cv_vector_store)

print('⚡ Building Candidate Knowledge Base...')
cv_index = VectorStoreIndex(
    cv_nodes,
    storage_context=cv_storage_context,
    show_progress=True,
)
cv_retriever = VectorIndexRetriever(index=cv_index, similarity_top_k=5)

print(f'\n✅ Candidate KB ready! Total vectors: {cv_collection.count()}')

📖 Loading candidate CV documents...
✅ Loaded 4 CV document(s)
✅ Created 4802 chunks from CV documents
⚡ Building Candidate Knowledge Base...


Generating embeddings: 100%|██████████| 706/706 [05:46<00:00,  2.04it/s]



✅ Candidate KB ready! Total vectors: 4802


In [30]:
# ============================================================
# CELL 12.2 — Load or Build Candidate Knowledge Base in ChromaDB
# ============================================================

print("\nLoading candidate CV documents...")

cv_reader = SimpleDirectoryReader(
    input_dir="./samplecv",
    required_exts=[".txt", ".pdf", ".docx"],
    recursive=True,
)
cv_documents = cv_reader.load_data()
print("Loaded", len(cv_documents), "CV document(s)")

# Chunk documents
cv_splitter = SentenceSplitter(chunk_size=256, chunk_overlap=32)
cv_nodes = cv_splitter.get_nodes_from_documents(cv_documents)
print("Created", len(cv_nodes), "chunks from CV documents")

CHROMA_DB_PATH = "./chroma_db"
CHROMA_COLLECTION = "candidate_kb"

chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

# Try loading existing collection
cv_collection = chroma_client.get_or_create_collection(
    name=CHROMA_COLLECTION,
    metadata={"hnsw:space": "cosine"}
)

existing_vectors = cv_collection.count()
print("Existing vectors in ChromaDB:", existing_vectors)

# ------------------------------------------------------------
# If vectors exist → load existing index
# If empty → build new index
# ------------------------------------------------------------

if existing_vectors > 0:
    print("Loading existing Candidate Knowledge Base...")
    cv_vector_store = ChromaVectorStore(chroma_collection=cv_collection)
    cv_storage_context = StorageContext.from_defaults(vector_store=cv_vector_store)
    cv_index = VectorStoreIndex.from_vector_store(
        vector_store=cv_vector_store,
        storage_context=cv_storage_context
    )
else:
    print("Building new Candidate Knowledge Base...")
    cv_vector_store = ChromaVectorStore(chroma_collection=cv_collection)
    cv_storage_context = StorageContext.from_defaults(vector_store=cv_vector_store)

    cv_index = VectorStoreIndex(
        cv_nodes,
        storage_context=cv_storage_context,
        show_progress=True,
    )

cv_retriever = VectorIndexRetriever(index=cv_index, similarity_top_k=5)

print("\nCandidate KB ready! Total vectors:", cv_collection.count())



Loading candidate CV documents...
Loaded 4 CV document(s)
Created 4802 chunks from CV documents
Existing vectors in ChromaDB: 4802
Loading existing Candidate Knowledge Base...

Candidate KB ready! Total vectors: 4802


In [31]:
# ============================================================
# CELL 12.3 — Resume Builder State & Helper
# ============================================================

class ResumeState(TypedDict):
    # Input
    job_title: str
    target_role: str
    # Planning
    resume_plan: dict
    sections_to_generate: List[str]
    # Generation & Validation
    current_section: str
    generated_sections: dict
    validation_results: dict
    retry_counts: dict
    # Assembly
    final_resume: str
    quality_report: str

def retrieve_from_cv_kb(query: str, top_k: int = 5) -> str:
    """Retrieve relevant CV content from ChromaDB."""
    nodes = cv_retriever.retrieve(query)
    texts = [n.text for n in nodes[:top_k]]
    return '\n\n'.join(texts)

def llm_generate(prompt: str) -> str:
    """Call LLM and return text."""
    resp = llm.complete(prompt)
    return getattr(resp, 'text', str(resp)).strip()

print('✅ ResumeState and helpers defined')

✅ ResumeState and helpers defined


In [21]:
# ============================================================
# CELL 12.4 — Agent 1: Resume Planner Agent
# ============================================================

RESUME_SECTIONS = [
    'Professional Summary',
    'Technical Skills',
    'Work Experience',
    'Projects',
    'Education',
    'Certifications'
]

def resume_planner_agent(state: ResumeState) -> ResumeState:
    print('\n📋 [PLANNER] Analyzing candidate profile...')

    # Retrieve overview from KB
    overview = retrieve_from_cv_kb('professional summary experience skills education', top_k=3)

    prompt = f"""
You are a Senior Resume Architect. Analyze the candidate's profile and create a resume generation plan.

Target Role: {state['target_role']}

Candidate Profile Overview:
{overview}

Create a JSON plan for these resume sections: {RESUME_SECTIONS}

For each section, provide:
- objective: what this section should achieve
- rag_query: the query to retrieve relevant content from the knowledge base
- expected_length: short/medium/long
- quality_requirements: key quality criteria

Return ONLY valid JSON, no preamble.
Format:
{{
  "Professional Summary": {{
    "objective": "...",
    "rag_query": "...",
    "expected_length": "short",
    "quality_requirements": "..."
  }},
  ...
}}
    """

    plan_text = llm_generate(prompt)

    # Parse JSON safely
    try:
        # Strip markdown fences if present
        clean = plan_text.replace('```json', '').replace('```', '').strip()
        resume_plan = json.loads(clean)
    except Exception:
        # Fallback: build default plan
        resume_plan = {
            s: {
                'objective': f'Generate {s} section',
                'rag_query': s.lower(),
                'expected_length': 'medium',
                'quality_requirements': 'Professional tone, quantified achievements'
            } for s in RESUME_SECTIONS
        }

    print(f'   ✅ Plan created for {len(resume_plan)} sections')
    for sec in resume_plan:
        print(f'      📌 {sec}')

    return {
        **state,
        'resume_plan': resume_plan,
        'sections_to_generate': list(resume_plan.keys()),
        'generated_sections': {},
        'validation_results': {},
        'retry_counts': {s: 0 for s in resume_plan.keys()},
    }

print('✅ Resume Planner Agent defined')

✅ Resume Planner Agent defined


In [22]:
# ============================================================
# CELL 12.5 — Agent 2: Section Content Agent
# ============================================================

def section_content_agent(state: ResumeState) -> ResumeState:
    sections = state['sections_to_generate']
    plan = state['resume_plan']
    generated = dict(state['generated_sections'])
    retry_counts = dict(state['retry_counts'])
    validation_results = dict(state['validation_results'])

    print(f'\n✍️  [SECTION AGENT] Generating all {len(sections)} sections...')

    for section in sections:
        section_plan = plan.get(section, {})
        rag_query = section_plan.get('rag_query', section)
        objective = section_plan.get('objective', f'Generate {section}')
        quality_req = section_plan.get('quality_requirements', 'Professional and quantified')

        # Retrieve relevant content
        context = retrieve_from_cv_kb(rag_query)

        max_attempts = 3  # 1 initial + 2 retries

        for attempt in range(max_attempts):
            print(f'   🔄 {section} — Attempt {attempt+1}/{max_attempts}')

            retry_instruction = ''
            if attempt > 0 and section in validation_results:
                prev_feedback = validation_results[section].get('feedback', [])
                retry_instruction = f'\nPrevious attempt failed. Fix these issues: {prev_feedback}'

            prompt = f"""
You are a Professional Resume Writer. Generate the "{section}" section of a resume.

Objective: {objective}
Target Role: {state['target_role']}
Quality Requirements: {quality_req}
{retry_instruction}

Candidate Information from Knowledge Base:
{context}

Instructions:
- Use strong action verbs
- Quantify achievements with numbers/percentages where possible
- Keep professional ATS-friendly tone
- Do NOT include the section header in the output
- Be specific and evidence-based

Generate the {section} section content now:
            """

            content = llm_generate(prompt)
            generated[section] = content
            retry_counts[section] = attempt

            # Validate inline
            val_result = validate_section(section, content, quality_req, attempt)
            validation_results[section] = val_result

            if val_result['status'] == 'PASS':
                print(f'      ✅ {section} PASSED (score: {val_result["score"]})')
                break
            else:
                print(f'      ❌ {section} FAILED (score: {val_result["score"]}) — {val_result["feedback"]}')
                if attempt == max_attempts - 1:
                    print(f'      ⚠️  Max retries reached for {section}. Storing as FAILED_AFTER_RETRIES.')
                    validation_results[section]['status'] = 'FAILED_AFTER_RETRIES'
                    validation_results[section]['attempts'] = attempt + 1

    return {
        **state,
        'generated_sections': generated,
        'validation_results': validation_results,
        'retry_counts': retry_counts,
    }

print('✅ Section Content Agent defined')

✅ Section Content Agent defined


In [23]:
# ============================================================
# CELL 12.6 — Agent 3: Validation Agent
# ============================================================

def validate_section(section: str, content: str, quality_req: str, attempt: int) -> dict:
    """
    Strict validation agent. Returns PASS/FAIL with score and feedback.
    Intentionally strict on first attempt to demonstrate retry loops.
    """
    prompt = f"""
You are a strict Resume Validation Agent. Evaluate the resume section below.

Section: {section}
Quality Requirements: {quality_req}

Content to validate:
{content}

Validate against these criteria:
1. Contains quantified achievements (numbers, percentages, scale)
2. Uses strong action verbs (Led, Architected, Built, Reduced, Increased, etc.)
3. Professional ATS-friendly tone and language
4. Relevant and specific (not generic filler)
5. Appropriate length and completeness

Score from 0-100. PASS if score >= 70, FAIL if < 70.

Return ONLY valid JSON:
{{
  "status": "PASS" or "FAIL",
  "score": <number 0-100>,
  "feedback": ["issue 1", "issue 2"] or []
}}
    """

    resp_text = llm_generate(prompt)

    try:
        clean = resp_text.replace('```json', '').replace('```', '').strip()
        result = json.loads(clean)
        # Ensure required fields
        if 'status' not in result:
            result['status'] = 'PASS' if result.get('score', 0) >= 70 else 'FAIL'
        return result
    except Exception:
        # Fallback: assume pass if content is substantial
        score = 75 if len(content) > 200 else 50
        return {
            'status': 'PASS' if score >= 70 else 'FAIL',
            'score': score,
            'feedback': [] if score >= 70 else ['Content too short or missing quantified achievements']
        }

print('✅ Validation Agent defined')


✅ Validation Agent defined


In [24]:
import os
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["PYTHONLEGACYWINDOWSSTDIO"] = "utf-8"


In [25]:
# ============================================================
# CELL 12.7 — Agent 4: Resume Assembly Agent
# ============================================================

def resume_assembly_agent(state: ResumeState) -> ResumeState:
    print('\n📄 [ASSEMBLY] Assembling final resume...')

    sections = state['sections_to_generate']
    generated = state['generated_sections']
    validation = state['validation_results']

    # Build resume in markdown format
    resume_lines = []
    resume_lines.append('# usama wahab khan')
    resume_lines.append('**Senior Software Engineer & AI/ML Specialist**')
    resume_lines.append('')
    resume_lines.append(' usama wahab khan |  linkedin.com/in/usamawahabkhan |  github.com/usamawahabkhan |  Dubai, UAE')
    resume_lines.append('')
    resume_lines.append('---')
    resume_lines.append('')

    passed = 0
    failed = 0

    for section in sections:
        content = generated.get(section, '[Section not generated]')
        val = validation.get(section, {})
        status = val.get('status', 'UNKNOWN')

        # Always include all sections (even failed ones, marked)
        resume_lines.append(f'## {section}')
        resume_lines.append('')
        resume_lines.append(content)
        resume_lines.append('')

        if 'FAIL' in status:
            resume_lines.append(f'>  *Validation Note: This section did not fully pass validation (score: {val.get("score", "N/A")})*')
            resume_lines.append('')
            failed += 1
        else:
            passed += 1

        resume_lines.append('---')
        resume_lines.append('')

    final_resume = '\n'.join(resume_lines)

    # Save to file
    with open('final_resume.md', 'w') as f:
        f.write(final_resume)

    print(f'   ✅ Resume assembled: {passed} passed, {failed} failed sections')
    print(f'   💾 Saved to: final_resume.md')

    return {**state, 'final_resume': final_resume}

print('✅ Resume Assembly Agent defined')

✅ Resume Assembly Agent defined


In [26]:
# ============================================================
# CELL 12.8 — Agent 5: Quality Report Agent
# ============================================================

def quality_report_agent(state: ResumeState) -> ResumeState:
    print('\n📊 [REPORT] Generating quality report...')

    sections = state['sections_to_generate']
    validation = state['validation_results']
    retry_counts = state['retry_counts']

    passed_sections = [s for s, v in validation.items() if v.get('status') == 'PASS']
    failed_sections = [s for s, v in validation.items() if 'FAIL' in v.get('status', '')]
    total_retries = sum(retry_counts.values())

    avg_score = 0
    scores = [v.get('score', 0) for v in validation.values() if 'score' in v]
    if scores:
        avg_score = sum(scores) / len(scores)

    report_lines = [
        '# Resume Generation Quality Report',
        '',
        f'**Generated for:** {state["target_role"]}',
        '',
        '---',
        '',
        '## Generation Summary',
        '',
        f'| Metric | Value |',
        f'|--------|-------|',
        f'| Total Sections | {len(sections)} |',
        f'| Passed Sections | {len(passed_sections)} |',
        f'| Failed Sections | {len(failed_sections)} |',
        f'| Total Retries | {total_retries} |',
        f'| Average Validation Score | {avg_score:.1f}/100 |',
        '',
        '---',
        '',
        '## Section-Level Validation Results',
        '',
    ]

    for section in sections:
        val = validation.get(section, {})
        status = val.get('status', 'UNKNOWN')
        score = val.get('score', 'N/A')
        feedback = val.get('feedback', [])
        attempts = retry_counts.get(section, 0) + 1
        emoji = 'pass' if status == 'PASS' else 'fail'

        report_lines.append(f'### {emoji} {section}')
        report_lines.append(f'- **Status:** `{status}`')
        report_lines.append(f'- **Score:** {score}/100')
        report_lines.append(f'- **Attempts:** {attempts}')
        if feedback:
            report_lines.append(f'- **Feedback:**')
            for fb in feedback:
                report_lines.append(f'  - {fb}')
        report_lines.append('')

    report_lines += [
        '---',
        '',
        '## Improvement Suggestions',
        '',
        '1. Add more quantified metrics to all bullet points (revenue, scale, % improvement)',
        '2. Ensure each role in Work Experience has 3-5 strong bullet points',
        '3. Tailor skills section to match target job description keywords',
        '4. Projects section should include business impact, not just technical stack',
        '5. Professional Summary should be reviewed and personalized for each application',
        '',
        '---',
        '*Report generated by RAG-Powered Resume Builder — LangGraph Multi-Agent System*'
    ]

    quality_report = '\n'.join(report_lines)

    with open('resume_generation_report.md', 'w') as f:
        f.write(quality_report)

    print(f'   ✅ Quality report saved to: resume_generation_report.md')
    print(quality_report[:800])

    return {**state, 'quality_report': quality_report}

print('✅ Quality Report Agent defined')

✅ Quality Report Agent defined


In [32]:
# ============================================================
# CELL 12.9 — Build Resume Builder LangGraph Workflow
# ============================================================

resume_workflow = StateGraph(ResumeState)

resume_workflow.add_node('planner', resume_planner_agent)
resume_workflow.add_node('section_generator', section_content_agent)
resume_workflow.add_node('assembler', resume_assembly_agent)
resume_workflow.add_node('reporter', quality_report_agent)

resume_workflow.set_entry_point('planner')
resume_workflow.add_edge('planner', 'section_generator')
resume_workflow.add_edge('section_generator', 'assembler')
resume_workflow.add_edge('assembler', 'reporter')

resume_app = resume_workflow.compile()

print('✅ Resume Builder LangGraph workflow compiled!')
print('\nWorkflow:')
print('  planner → section_generator (with validation loops) → assembler → reporter')

✅ Resume Builder LangGraph workflow compiled!

Workflow:
  planner → section_generator (with validation loops) → assembler → reporter


In [52]:
# ============================================================
# CELL 12.10 — Run the Resume Builder
# ============================================================

print('🚀 Starting RAG-Powered Resume Builder...')
print('=' * 65)

initial_state: ResumeState = {
    'job_title': 'Senior Software Engineer',
    'target_role': 'Senior Software Engineer / AI Platform Engineer at a high-growth tech company',
    'resume_plan': {},
    'sections_to_generate': [],
    'current_section': '',
    'generated_sections': {},
    'validation_results': {},
    'retry_counts': {},
    'final_resume': '',
    'quality_report': '',
}

final_state = resume_app.invoke(initial_state)

print('\n' + '=' * 65)
print('🏁 RESUME BUILDER COMPLETE!')
print('=' * 65)
print('\n📄 Output files:')
print('   final_resume.md')
print('   resume_generation_report.md')

🚀 Starting RAG-Powered Resume Builder...

📋 [PLANNER] Analyzing candidate profile...
   ✅ Plan created for 6 sections
      📌 Professional Summary
      📌 Technical Skills
      📌 Work Experience
      📌 Projects
      📌 Education
      📌 Certifications

✍️  [SECTION AGENT] Generating all 6 sections...
   🔄 Professional Summary — Attempt 1/3
      ✅ Professional Summary PASSED (score: 92)
   🔄 Technical Skills — Attempt 1/3
      ❌ Technical Skills FAILED (score: 65) — ['The section lacks quantified achievements and specific metrics.', 'Some skills are listed without strong action verbs.']
   🔄 Technical Skills — Attempt 2/3
      ✅ Technical Skills PASSED (score: 92)
   🔄 Work Experience — Attempt 1/3
      ✅ Work Experience PASSED (score: 92)
   🔄 Projects — Attempt 1/3
      ✅ Projects PASSED (score: 92)
   🔄 Education — Attempt 1/3
      ✅ Education PASSED (score: 85)
   🔄 Certifications — Attempt 1/3
      ✅ Certifications PASSED (score: 85)

📄 [ASSEMBLY] Assembling final resume..

In [34]:
# ============================================================
# CELL 12.10.2 — Run the Resume Builder
# ============================================================

print('🚀 Starting RAG-Powered Resume Builder...')
print('=' * 65)

initial_state: ResumeState = {
    'job_title': 'Investment Analyst',
    'target_role': 'Investment Analyst at a leading financial services firm blackrock',
    'resume_plan': {},
    'sections_to_generate': [],
    'current_section': '',
    'generated_sections': {},
    'validation_results': {},
    'retry_counts': {},
    'final_resume': '',
    'quality_report': '',
}

final_state = resume_app.invoke(initial_state)

print('\n' + '=' * 65)
print('🏁 RESUME BUILDER COMPLETE!')
print('=' * 65)
print('\n📄 Output files:')
print('   final_resume.md')
print('   resume_generation_report.md')

🚀 Starting RAG-Powered Resume Builder...

📋 [PLANNER] Analyzing candidate profile...
   ✅ Plan created for 6 sections
      📌 Professional Summary
      📌 Technical Skills
      📌 Work Experience
      📌 Projects
      📌 Education
      📌 Certifications

✍️  [SECTION AGENT] Generating all 6 sections...
   🔄 Professional Summary — Attempt 1/3


KeyboardInterrupt: 

In [53]:
# ============================================================
# CELL 12.11 — Display Final Resume
# ============================================================

print('\n' + '='*65)
print('📄 FINAL RESUME')
print('='*65)
print(final_state['final_resume'])


📄 FINAL RESUME
# usama wahab khan
**Senior Software Engineer & AI/ML Specialist**

 usama wahab khan |  linkedin.com/in/usamawahabkhan |  github.com/usamawahabkhan |  Dubai, UAE

---

## Professional Summary

Driven Senior Software Engineer with over 8 years of experience in developing and maintaining AI platforms. Spearheaded the development of machine learning models that improved predictive accuracy by 25%, resulting in a 20% increase in operational efficiency. Proven track record of driving innovation and delivering high-quality solutions across multiple industries. Expert in leveraging cutting-edge technologies to solve complex problems and deliver scalable, secure, and user-friendly applications.

---

## Technical Skills

- **Programming Languages:** Proficient in Python (5+ years), Java (4+ years), C++ (3+ years), and JavaScript (4+ years). Developed and maintained over 10 large-scale applications using Python, reducing development time by 25%.

- **Frameworks & Libraries:** E

In [54]:
# ============================================================
# CELL 12.12 — Display Quality Report
# ============================================================

print('\n' + '='*65)
print('📊 QUALITY REPORT')
print('='*65)
print(final_state['quality_report'])


📊 QUALITY REPORT
# Resume Generation Quality Report

**Generated for:** Senior Software Engineer / AI Platform Engineer at a high-growth tech company

---

## Generation Summary

| Metric | Value |
|--------|-------|
| Total Sections | 6 |
| Passed Sections | 6 |
| Failed Sections | 0 |
| Total Retries | 1 |
| Average Validation Score | 89.7/100 |

---

## Section-Level Validation Results

### pass Professional Summary
- **Status:** `PASS`
- **Score:** 92/100
- **Attempts:** 1

### pass Technical Skills
- **Status:** `PASS`
- **Score:** 92/100
- **Attempts:** 2

### pass Work Experience
- **Status:** `PASS`
- **Score:** 92/100
- **Attempts:** 1

### pass Projects
- **Status:** `PASS`
- **Score:** 92/100
- **Attempts:** 1

### pass Education
- **Status:** `PASS`
- **Score:** 85/100
- **Attempts:** 1

### pass Certifications
- **Status:** `PASS`
- **Score:** 85/100
- **Attempts:** 1

---

## Improvement Suggestions

1. Add more quantified metrics to all bullet points (revenue, scale, % 